# 🌊 AquaGuard AI — Day 15: Anomaly Detection Baselines
- **Date:** 03 Sep 2026
- **Phase:** Phase 3: Modeling & Validation
- **Objective:** Establish statistical baselines (Z-score, IQR fences) and fit unsupervised Isolation Forest on continuous sensor streams.
- **Inputs Consumed:** `data/processed/dataset_v2_features.parquet`
- **Outputs Produced:** `artifacts/day15/anomaly_baseline_eval.json`, `artifacts/day15/anomaly_overlay_plot.png`
- **Deliverable Satisfied:** Anomaly detection rate, score stability, and visual sensor telemetry overlay.

In [1]:
import os, sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").resolve().name == "Water Potability" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_parquet, save_artifact
from src.anomaly import ZScoreAnomalyDetector, IQRAnomalyDetector, IsolationForestAnomalyDetector, evaluate_anomaly_detection
from src.plotting import plot_anomalies

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day15")

# Load Dataset V2
feat_df = load_parquet("data/processed/dataset_v2_features.parquet", required_by_day="Day 13")
sensor_cols = config["track_a"]["sensor_columns"]
X = feat_df[sensor_cols].values
print(f"Loaded feature dataset: {feat_df.shape}")

Loaded feature dataset: (612, 149)


## 1. Fit Anomaly Baseline Models
Train Z-score statistical detector ($3.5\sigma$), Tukey IQR fences, and unsupervised Isolation Forest.

In [2]:
# 1. Z-Score Baseline
z_det = ZScoreAnomalyDetector(threshold=config["track_a"]["anomaly"]["z_score_threshold"])
z_det.fit(X)
z_eval = evaluate_anomaly_detection(z_det, X)

# 2. IQR Baseline
iqr_det = IQRAnomalyDetector(multiplier=1.5)
iqr_det.fit(X)
iqr_eval = evaluate_anomaly_detection(iqr_det, X)

# 3. Isolation Forest
iso_det = IsolationForestAnomalyDetector(
    contamination=config["track_a"]["anomaly"]["isolation_forest_contamination"],
    random_state=config["seed"]
)
iso_det.fit(X)
iso_eval = evaluate_anomaly_detection(iso_det, X)

comparison = {
    "Z_Score_Baseline": z_eval,
    "IQR_Fence_Baseline": iqr_eval,
    "Isolation_Forest": iso_eval
}

save_artifact(comparison, "artifacts/day15/anomaly_baseline_eval.json")
print("Anomaly detection evaluation results:")
for k, v in comparison.items():
    print(f"  {k}: {v}")

Anomaly detection evaluation results:
  Z_Score_Baseline: {'detection_rate': 0.0114, 'total_anomalies': 7, 'score_std': 1.2385}
  IQR_Fence_Baseline: {'detection_rate': 0.0229, 'total_anomalies': 14, 'score_std': 0.0}
  Isolation_Forest: {'detection_rate': 0.0212, 'total_anomalies': 13, 'score_std': 0.0345}


## 2. Visualize Telemetry Stream Anomaly Highlights
Overlay detected anomalies onto Dissolved Oxygen and Turbidity telemetry.

In [3]:
anom_flags = iso_det.predict(X)

plot_anomalies(
    feat_df,
    sensor="dissolved_oxygen",
    anomaly_flags=anom_flags,
    timestamp_col=config["track_a"]["timestamp_column"],
    title="Isolation Forest Anomaly Flags",
    save_path="artifacts/day15/anomaly_overlay_plot.png"
)
print("Anomaly overlay plot saved to artifacts/day15/anomaly_overlay_plot.png")

Anomaly overlay plot saved to artifacts/day15/anomaly_overlay_plot.png


## 📌 Day 15 Execution Findings & Sign-Off
- **Key Finding:** Isolation Forest flagged ~2% of extreme multi-dimensional deviations with high score stability.
- **Artifacts Saved:** `artifacts/day15/anomaly_baseline_eval.json`, `artifacts/day15/anomaly_overlay_plot.png`.
- **Handoff:** Ready for Day 16 multi-step time-series forecasting baselines.